
# Home-made Deepwiki project

This is a small one-week project to learn how to build intelligent systems that can understand and interact with your data. If you're familiar with DeepWiki, it's similar, but tailored to your GitHub repository.

## Day1: Understanding Frontmatter
We will also need a library for parsing frontmatter - a popular documentation format commonly used for modern frameworks like Jekyll, Hugo, and Next.js.

It looks like this:
```
---
title: "Getting Started with AI"
author: "John Doe"
date: "2024-01-15"
tags: ["ai", "machine-learning", "tutorial"]
difficulty: "beginner"
---

# Getting Started with AI

This is the main content of the document written in **Markdown**.

You can include code blocks, links, and other formatting here.

```

In [7]:
!pip install python-frontmatter

## getting data from github repo (as zip download)

In [25]:
import io
import zipfile
import requests
import frontmatter

In [9]:
url = 'https://codeload.github.com/TacticalNuclearRaccoon/code_instructions/zip/refs/heads/main'
resp = requests.get(url)

In [10]:
resp

<Response [200]>

In [11]:
#Now we process the zip file in memory without saving it to disk:
repository_data = []

# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files
    if not filename.endswith('.md'):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()


In [12]:
print(repository_data[1])

{'content': '# Agent2Agent (A2A) Protocol Development Assistant\n\nExpert A2A protocol advisor for robust, efficient, interoperable AI agent communication systems.\n\n## A2A Fundamentals & Design\n\n1. **Protocol Understanding**: Explain A2A purpose, components, and communication between agentic applications.\n2. **Core Actors**: Guide on A2A Client (Client Agent), A2A Server (Remote Agent), and End User roles.\n3. **Discovery**: Implement Agent Cards at well-known URLs exposing capabilities, endpoints, and auth requirements.\n4. **Task Management**: Handle lifecycle states, message exchange, content delivery (Parts), and artifacts.\n5. **Interaction Modes**: Implement synchronous requests, Server-Sent Events for streaming, and push notifications.\n6. **Stateful Operations**: Maintain state across task execution and multi-turn interactions.\n\n## Implementation Best Practices\n\n1. **Idiomatic Code**: Follow target language conventions while maintaining A2A compliance.\n2. **Error Hand

**For most documentation repos we also need ```.mdx``` files (React markdown), so we can modify the code like this:**

In [13]:
for file_info in zf.infolist():
    filename = file_info.filename.lower()

    if not (filename.endswith('.md') or filename.endswith('.mdx')):
        continue

    # rest remains the same...


## Complete implementation

In [26]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md')
            or filename_lower.endswith('.mdx')):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    zf.close()
    return repository_data


As an example of working with large amounts of documantation, I wanted to use the docs from EvidentlyAI but I also wanted to keep the process offline (no API calls to foundation model providers like openai for example) I had to change plans. The cloud notebook I used wan limited to 15GB of GPU ram (T4 gpu). Therefore it crashed at the intelligent chunking attempt. Therefore I use only one of my repos with a modest amount of 20 or so docs for proof of concept purposes.

In [15]:
code_inst = read_repo_data('TacticalNuclearRaccoon', 'code_instructions')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"code instructions: {len(code_inst)}")
print(f"Evidently documents: {len(evidently_docs)}")


code instructions: 22
Evidently documents: 95


In [16]:
evidently_docs

[{'title': 'Create Plant',
  'openapi': 'POST /plants',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/create.mdx'},
 {'title': 'Delete Plant',
  'openapi': 'DELETE /plants/{id}',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/delete.mdx'},
 {'title': 'Get Plants',
  'openapi': 'GET /plants',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/get.mdx'},
 {'title': 'Introduction',
  'description': 'Example section for showcasing API endpoints',
  'content': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"

## Day 2: Chunking and Intelligent Processing for Data

When the information amount is small, the proceess above is sufficient for applications like a FAQ database. But for larger repos like Evidently AI documentation where the documents are quite large we risk overwhelming the LLM.

### How to prepare large documents before using them:

Large documents create several problems:

* Token limits: Most LLMs have maximum input token limits
* Cost: Longer prompts cost more money
* Performance: LLMs perform worse with very long contexts
* Relevance: Not all parts of a long document are relevant to a specific question

So we need to split documents into smaller subdocuments. For AI applications like RAG this process is referred to as "chunking."

There are several ways to perform chunking:

* Simple character-based chunking
* Paragraph and section-based chunking
* Intelligent chunking with LLM



### 1- Simple chunking

In [17]:
code_inst[5]

{'content': '# Python API Development Assistant\n\nExpert Python API advisor focusing on robust, efficient, well-documented API systems using modern frameworks and best practices.\n\n## Framework Selection & Architecture\n\n1. **Frameworks**: FastAPI (performance, types, docs), Flask (lightweight, flexible), Django REST (complex, ORM), Falcon (minimalist), Starlette (ASGI).\n\n2. **Architecture**: Apply domain-driven design; structure with src-layout; use ABCs/protocols; implement dependency injection.\n\n## Protocol Implementations\n\n1. **REST**: Use type annotations with Pydantic; implement OpenAPI; use context managers; design proper endpoints.\n\n2. **A2A Protocol**: Create async clients with httpx; implement SSE; build Agent Cards with Pydantic; manage with asyncio.\n\n3. **MCP**: Use context management; implement efficient window handling; create streaming responses; build type-safe schemas.\n\n4. **GraphQL**: Implement with Strawberry/Ariadne/Graphene; design schema-first or co

The content field is 21,712 characters long. The simplest thing we can do is cut it into pieces of equal length. For example, for size of 2000 characters, we will have:

* Chunk 1: 0..2000
* Chunk 2: 2000..4000
* Chunk 3: 4000..6000

And so on.

However, this approach has disadvantages:

* Context loss: Important information might be split in the middle
* Incomplete sentences: Chunks might end mid-sentence
* Missing connections: Related information might end up in different chunks

That's why, in practice, we usually make sure there's overlap between chunks. For size 2000 and overlap 1000, we will have:

* Chunk 1: 0..2000
* Chunk 2: 1000..3000
* Chunk 3: 2000..4000
...

This is better for AI because:

* Continuity: Important information isn't lost at chunk boundaries
* Context preservation: Related sentences stay together in at least one chunk
* Better search: Queries can match information even if it spans chunk boundaries

This approach is known as the "sliding window" method.


In [39]:
# implementing slinding window
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

If we apply this to document 45. This gives us 21 chunks:

* 0..2000
* 1000..3000
* ...
* 19000..21000
* 20000..21712


In [40]:
# processing all the documents in my repo
code_inst_chunks = []

for doc in code_inst:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    # Add a print statement to observe if chunks are being generated
    print(f"Processing '{doc_copy.get('filename', 'Unknown')}', generated {len(chunks)} chunks.")
    for chunk in chunks:
        chunk.update(doc_copy)
    code_inst_chunks.extend(chunks)
print(f"Total chunks after processing: {len(code_inst_chunks)}")

Processing 'code_instructions-main/README.md', generated 5 chunks.
Processing 'code_instructions-main/protocols/a2a.instructions.md', generated 3 chunks.
Processing 'code_instructions-main/protocols/agent.instructions.md', generated 5 chunks.
Processing 'code_instructions-main/protocols/mcp.instructions.md', generated 3 chunks.
Processing 'code_instructions-main/protocols/rest_api.instructions.md', generated 2 chunks.
Processing 'code_instructions-main/python/python_api_development.instructions.md', generated 2 chunks.
Processing 'code_instructions-main/python/python_best_practices.instructions.md', generated 2 chunks.
Processing 'code_instructions-main/python/python_data_science.instructions.md', generated 2 chunks.
Processing 'code_instructions-main/python/python_guide.instructions.md', generated 2 chunks.
Processing 'code_instructions-main/python/python_scripting.instructions.md', generated 2 chunks.
Processing 'code_instructions-main/python/python_testing_quality.instructions.md', 

In [41]:
len(code_inst_chunks)

59

In [19]:
#processing documents in evidently AI repo
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)

Note that we use copy() and pop() operations:

* doc.copy() creates a shallow copy of the document dictionary
* doc_copy.pop('content') removes the 'content' key and returns its value
* This way we preserve the original dictionary keys that we can use later in the chunks.

Thus, we obtain 575 chunks from 95 documents.

We can play with the parameters by including more or less content. 2000 characters is usually good enough for RAG applications.

There are some alternative approaches:

* Token-based chunking: You first tokenize the content (turn it into a sequence of words) and then do a sliding window over tokens
  * Advantages: More precise control over LLM input size
  * Disadvantages: Doesn't work well for documents with code
* Paragraph splitting: Split by paragraphs
* Section splitting: Split by sections
* AI-powered splitting: Let AI split the text intelligently


### Splitting by paragraphs and sections

In [21]:
len(code_inst)

22

In [42]:
# splitting by paragraphs (my repo)
import re
text = code_inst[20]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [23]:
# for evidentlyAi docs
text = evidently_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

We use \n\s*\n regex pattern for splitting:

* \n matches a newline
* \s* matches zero or more whitespace characters
* \n matches another newline
* So \n\s*\n matches two newlines with optional whitespace between them

This works well for literature, but it doesn't work well for documents. Most paragraphs in technical documentation are very short.

You can combine sliding window and paragraph splitting for more intelligent processing. We won't do it here, but it's a good exercise to try.

Let's now look at section splitting. Here, we take advantage of the documents' structure. Markdown documents have this structure:
```
# Heading 1
## Heading 2  
### Heading 3
```

In [38]:
# use regax to split by headers

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections

**Note:** This code may not work perfectly if we want to split by level 1 headings and have Python code with # comments. But in general, this is not a big problem for documentation.

In [25]:
# split by second level headers
sections = split_markdown_by_level(text, level=2)

In [43]:
# iterate over all the docs and create chunks (my repo)
code_inst_chunks = []

for doc in code_inst_chunks:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        code_inst_chunks.append(section_doc)

In [27]:
# in the example of avidently Ai
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)


Like previously, copy() creates a copy of the document metadata. pop('content') removes and returns the content. This way, each section gets the same metadata (title, description) as the original document.

### Intelligent chunking with LLM

In some cases, we want to be more intelligent with chunking. Instead of doing simple splits, we delegate this work to AI.

This makes sense when:

* Complex structure: Documents have complex, non-standard structure
* Semantic coherence: You want chunks that are semantically meaningful
* Custom logic: You need domain-specific splitting rules
* Quality over cost: You prioritize quality over processing cost

This costs money. In most cases, we don't need intelligent chunking.

Simple approaches are sufficient. Use intelligent chunking only when

* You already evaluated simpler methods and you can confirm that they produce poor results
* You have complex, unstructured documents
* Quality is more important than cost
* You have the budget for LLM processing (which is not my case so I will try a local model) **Caution:** This needs GPU otherwise takes ages to complete.   


In [3]:
# code was initially to work with openAI
# Transformed to work with Hugging Face Mistral 7B
# for my repo this is feasible for evidentlyAI has so much volumen that it still crushes...

from transformers import pipeline
from google.colab import userdata

# Retrieve Hugging Face API key from Colab secrets
# Make sure you have a secret named 'HUGGING_FACE_TOKEN' in your Colab notebook
HF_TOKEN = userdata.get('HF_TOKEN')

# Initialize the Hugging Face text generation pipeline
# Using Mistral-7B-Instruct-v0.2 as requested
# This will download the model the first time it's run
# Using device_map='auto' for automatic device placement (GPU if available, else CPU)
generator = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    token=HF_TOKEN,
    device_map="auto"
)

def llm(prompt, model="mistral:latest"): # The 'model' parameter will be ignored as the pipeline is fixed to Mistral 7B
    # Mistral instruction format for chat templates
    messages = [
        {"role": "user", "content": prompt}
    ]
    formatted_prompt = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Generate text using the Hugging Face pipeline
    outputs = generator(
        formatted_prompt,
        max_new_tokens=200, # Adjust as needed for response length
        do_sample=True,
        temperature=0.7,    # Adjust as needed for creativity
        top_k=50,           # Adjust as needed for diversity
        top_p=0.95,         # Adjust as needed for focus
        truncation=True     # Handle long inputs by truncating
    )

    # Extract the generated content
    # The output from the pipeline includes the prompt, so we remove it to get only the new generation
    generated_text = outputs[0]["generated_text"]
    response_content = generated_text.replace(formatted_prompt, "").strip()

    return response_content


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [31]:
#here is the code to use with openai (for processing evidentlyai docs)
from openai import OpenAI
from google.colab import userdata

# Retrieve OpenAI API key from Colab secrets
OPENAI_API_KEY = userdata.get('openai')

openai_client = OpenAI(api_key=OPENAI_API_KEY)

def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages
    )

    return response.choices[0].message.content

In [32]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()

The prompt asks the LLM to:

* Split the document logically (not just by length)
* Make sections self-contained
* Use a specific output format that's easy to parse

In [45]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [46]:
from tqdm.auto import tqdm

In [47]:
# apply intelligent chunking to all documents (my repo)
# this still took a lot of time for mistral7b for my 20 or so documents so for evidently don't even try
code_inst_chunks = []

for doc in tqdm(code_inst):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        code_inst_chunks.append(section_doc)

  0%|          | 0/22 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'top_k', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Bot

The intelligent chunking step above is the GPU/ram greedy step that crushed with the evidently ai docs. But it should work fine with an api call to a powerful foundation model provider ;)

In [ ]:
# evidently chunks intelligent chunking with openai
evidently_chunks = []

for doc in tqdm(evidently_docs):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)


(Ok for openai, it exceeds my quota so nevermind I guess...)

## Day 3 : Add search

So far we focused on data preparation. Before we can use data for AI agents, we need to prepare it properly. This is basically the core of knowledge engineering.

Now it's time to use this data. We will index this data by putting it inside a search engine. This allows us to quickly find relevant information when users ask questions.

In particular, we will:
* Build a lexical search for exact matches and keywords
* Implement semantic search using embeddings
* Combine them with a hybrid search

At the end this day we'll have a working search system can be queried about a project. This search engine can be used later by the AI agent to look up user questions in the database.

### 1. Text search

The simplest type of search is a text search. Suppose we build a Q&A system for courses (using the FAQ dataset). We want to find the answer to this question:
"What should be in a test dataset for AI evaluation?"

Text search works by finding all documents that contain at least one word from the query. The more words from the query that appear in a document, the more relevant that document is.

This is how modern search systems like Apache Solr or Elasticsearch work. They use indexes to efficiently search through millions of documents without having to scan each one individually.

We'll start with a simple in-memory text search. The engine we will use is called minsearch.

In [36]:
!pip install minsearch

In [ ]:
code_inst_chunks[0]

In [ ]:
# We will use it for chunked docs from my repo.
from minsearch import Index

index = Index(
    text_fields=["filename","section"],
    keyword_fields=[]
)

index.fit(code_inst_chunks)


Here we create an index that will search through four text fields: chunk content, title, description, and filename. The keyword_fields parameter is for exact matches (we don't need it for now).

For DataTalks FAQ repo it's similar, except we don't need to chunk the data.

In [30]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')

de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)

In [46]:
de_dtc_faq

[{'id': '9e508f2212',
  'question': 'Course: When does the course start?',
  'sort_order': 1,
  'content': "The next cohort starts January 12th, 2026. More info at [DTC](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\n- Register before the course starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD).\n- Join the [course Telegram channel with announcements](https://t.me/dezoomcamp).\n- Don’t forget to register in DataTalks.Club's Slack and [join](https://datatalks.club/docs/general/slack/) the channel.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/001_9e508f2212_course-when-does-the-course-start.md'},
 {'id': 'bfafa427b3',
  'question': 'Course: What are the prerequisites for this course?',
  'sort_order': 2,
  'content': 'To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experie

This is text search, also known as "lexical search". We look for exact matches between our query and the documents.

### 2. Vector search

Text search has limitations. Consider these two queries:
* "I just discovered the program, can I still enroll?"
* "I just found out about the course, can I still join?"

These ask the same question but share no common words (among important ones). Text search would fail to find relevant matches.
This is where embeddings help. Embeddings are numerical representations of text that capture semantic meaning. Words and phrases with similar meanings have similar embeddings, even if they use different words.

Vector search uses these embeddings to identify semantically similar documents, rather than just exact word matches.
For vector search, we need to turn our documents into vectors (embeddings).
We will use the sentence-transformers library for this purpose.


In [52]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The multi-qa-distilbert-cos-v1 model is trained explicitly for question-answering tasks. It creates embeddings optimized for finding answers to questions.

Other popular models include:
* all-MiniLM-L6-v2 - General-purpose, fast, and efficient
* all-mpnet-base-v2 - Higher quality, slower

In [53]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

We combine the question and answer text, then convert it to an embedding vector.

In [54]:
query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)

This is how we compute similarity between the query and document vectors:

In [55]:
similarity = v_query.dot(v_doc)

The dot product measures similarity between vectors.

Values closer to 1 indicate higher similarity, closer to 0 means lower similarity. This works because the model creates normalized embeddings where cosine similarity equals the dot product.

So we can create embeddings for all documents, then compute similarity between the query and each document to find the most similar ones.
This is what VectorSearch from minsearch does. Let's use it.

First, we turn our docs into embeddings. This process takes time, so we'll monitor progress with tqdm:

In [57]:
#from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for d in tqdm(de_dtc_faq):
    text = d['question'] + ' ' + d['content']
    v = embedding_model.encode(text)
    faq_embeddings.append(v)

faq_embeddings = np.array(faq_embeddings)

  0%|          | 0/498 [00:00<?, ?it/s]

We combine question and answer text for each FAQ entry. We convert the list to a NumPy array for efficient similarity computations.

Now let's use VectorSearch:

In [58]:
from minsearch import VectorSearch

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)

This creates a vector search index using our embeddings and original documents.

In [59]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)

We first create an embedding for our query (q), then search for similar document embeddings.
You can easily do the same with the Evidently docs (but only use the chunk field for embeddings):

In [60]:
code_ins_embeddings = []

for d in tqdm(code_inst_chunks):
    v = embedding_model.encode(d['filename'])
    code_ins_embeddings.append(v)

code_ins_embeddings = np.array(code_ins_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(code_ins_embeddings, code_inst_chunks)

  0%|          | 0/45 [00:00<?, ?it/s]

### 3. Hybrid search
Text search is fast and efficient. It works well for exact matches and specific terms, and requires no model inference. However, it misses semantically similar but differently worded queries and struggles to handle synonyms effectively.

Vector search captures semantic meaning and handles paraphrased questions. It works with synonyms and related concepts. But it may miss exact keyword matches.
Combining both approaches gives us the best of both worlds. This is known as "hybrid search."
The code is quite simple:

In [61]:
query = 'Can I join the course now?'

text_results = faq_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = faq_vindex.search(q, num_results=5)

final_results = text_results + vector_results

### 4. Putting this together
Our search is implemented!
But before we can use it in our agent, we need to organize the code. Let's put all the code into different functions:

def text_search(query):
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results

### 5. Selecting the best approach

Much like with chunking, you should always start with the simplest approach. For search, that's text search. It's faster, easier to debug, and works well for many use cases. Only add complexity when a simple text search isn't sufficient.

## Day 4: Agents and Tools

So far, we have done:
* Day 1: Downloaded the data from a GitHub repository
* Day 2: Processed it by chunking it where necessary
* Day 3: Indexed the data so it's searchable

Note that it took us quite a lot of time. This is not a coincidence. Data preparation is the most time-consuming and critical part of building AI agents. Without properly prepared, cleaned, and indexed data, even the most sophisticated agent will provide poor results.
Now it's time to create an AI agent that will use this data through the search engine that we created before.
This allows us to build context-aware agents. They can provide accurate, relevant answers based on your specific domain knowledge rather than just general training data.

In particular, we will:
* Learn what makes an AI system "agentic" through tool use
* Build an agent that can use the search function
* Use Pydantic AI to make it easier to implement agents

### 1. Tools and Agents

I have the impression that everyone has their own definition for an AI agent. But here is a simple one: an agent is an LLM that can not only generate texts, but also invoke tools. Tools are external functions that the LLM can call in order to retrieve information, perform calculations, or take actions.
In our case, the agent needs to answer our questions using the content of the GitHub repository. So, the tool (only one) is a search(query).
But first, let's consider a situation where we have no tools at all. This is not an agent, it's just an LLM that can generate texts. Access to tools is what makes agents "agentic".

Let's see the difference with an example.
We will try asking a question without giving the LLM access to search:

In [62]:
# I will modify this code as well to work woth hugging face mistral (since my openai quota is so low)
# No import openai needed as we are using the custom llm function
# No openai_client needed

user_prompt = "I just discovered the course, can I join now?"

# chat_messages is not directly used by the custom llm function,
# as the llm function constructs its own messages list internally.
# We will pass the user_prompt directly to the llm function.

response = llm(user_prompt) # Call the pre-defined llm function with the user_prompt

print(response) # The llm function directly returns the generated text

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Absolutely! Our courses have rolling enrollment, which means you can join at any time. Once you've signed up, you'll have access to all the course materials and can begin learning at your own pace. If you have any questions or need assistance getting started, please don't hesitate to contact us. We're here to help!

To get started, simply visit our website and select the course you're interested in. From there, you'll be able to sign up and make your payment. Once you've completed the registration process, you'll receive an email with instructions on how to access the course materials.

If you have any questions or need further assistance, please don't hesitate to contact us using the information provided on our website. We're here to help you every step of the way!


The response is generic. In our case, it's this:

“It depends on the course you're interested in. Many courses allow late enrollment, while others might have specific deadlines. I recommend checking the course's official website or contacting the instructor or administration for more details on joining.”

This answer is not really useful.

But if we let it invoke the search(query), the agent can give us a more useful answer.

Here's how the conversation would flow with our agent using the search tool:
* User: "I just discovered the course, can I join now?"
* Agent thinking: I can't answer this question, so I need to search for information about course enrollment and timing.
* Tool call: search("course enrollment join registration deadline")
* Tool response: (...search results...)
* Agent response: "Yes, you can still join the course even after the start date..."

We will now explore how to implement it with OpenAI.

### 2. Function Calling with OpenAI (or with another LLM)

One problem that we are facing now is that we have chosen to use mistral 7b model from Hugging Face but it doesn't support tool calling. The only way we can use it in our context is like below:

In [65]:
system_prompt = """
You are a helpful assistant for a course.
"""

question = "I just discovered the course, can I join now?"

# The llm function (Mistral 7B) is a simple text generation function
# and does not support the 'tools' argument or structured tool-call responses.
# It also does not directly process a separate 'system_prompt' in the same way.
# It expects a single prompt string as input.

response = llm(question)

# The 'response' object will now be a string, not an object with '.output' or tool calls.
# Subsequent cells in this section that expect tool call information will fail.

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The 'response' object will now be a string, not an object with '.output' or tool calls.

Subsequent cells in this section that expect tool call information will fail.

### Can we trick Mistral 7b to use Tool-like functions?

Let's fool around to see if we can simulate it via prompting: instruct the model to output JSON when it wants to call a tool, then parse that output yourself.

In [5]:
# Define your tools as a description the model can understand
tools_description = """
You have access to the following tools:
- text_search(query: str): Searches the course documentation for relevant information.

When you need to use a tool, respond ONLY with a JSON object like:
{"tool_call": {"name": "text_search", "arguments": {"query": "your search query"}}}

When you have enough information to answer, respond normally in plain text.
"""

system_prompt = f"""You are a helpful assistant for a course.
{tools_description}
"""

def build_prompt(system, messages):
    """Combine system prompt + chat history into a single prompt for Mistral."""
    full_messages = [{"role": "system", "content": system}] + messages
    return generator.tokenizer.apply_chat_template(
        full_messages, tokenize=False, add_generation_prompt=True
    )

# Simulated response object to mimic OpenAI's structure
class ToolCall:
    def __init__(self, name, arguments):
        self.name = name
        self.arguments = arguments  # dict

class Message:
    def __init__(self, content, tool_call=None):
        self.content = content
        self.tool_call = tool_call  # ToolCall or None

class Response:
    def __init__(self, message):
        self.output = [message]

def parse_response(raw_text):
    """Parse raw model output into a Response object."""
    raw_text = raw_text.strip()
    # Try to detect a tool call in the output
    json_match = re.search(r'\{.*"tool_call".*\}', raw_text, re.DOTALL)
    if json_match:
        try:
            parsed = json.loads(json_match.group())
            tc = parsed["tool_call"]
            tool_call = ToolCall(name=tc["name"], arguments=tc["arguments"])
            return Response(Message(content=None, tool_call=tool_call))
        except (json.JSONDecodeError, KeyError):
            pass
    # Plain text response
    return Response(Message(content=raw_text))

def llm_with_tools(messages, system_prompt):
    prompt = build_prompt(system_prompt, messages)
    outputs = generator(
        prompt,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        truncation=True
    )
    raw = outputs[0]["generated_text"].replace(prompt, "").strip()
    return parse_response(raw)

In [72]:
# now our agentic loop looks like this:
def run_agent(question, system_prompt, tools: dict):
    messages = [{"role": "user", "content": question}]

    while True:
        response = llm_with_tools(messages, system_prompt)
        message = response.output[0]

        if message.tool_call:
            # Execute the tool
            tool_name = message.tool_call.name
            tool_args = message.tool_call.arguments
            print(f"Calling tool: {tool_name} with args: {tool_args}")

            tool_result = tools[tool_name](**tool_args)

            # Feed result back into context
            messages.append({"role": "assistant", "content": f'[Tool call: {tool_name}({tool_args})]'}) # Corrected line
            messages.append({"role": "user", "content": f"Tool result: {tool_result}"})
        else:
            # Final answer
            return message.content

# Example usage
# Make sure the 'text_search' function is defined before this cell is run.
# The 'text_search_tool' variable is a *description* of the tool, not the tool itself.
# We need to pass the actual callable function 'text_search'.
result = run_agent(
    question="I just discovered the course, can I join now?",
    system_prompt=system_prompt,
    tools={"text_search": text_search} # Changed text_search_tool to text_search
)
print(result)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Calling tool: text_search with args: {'query': 'course registration'}
To join the course, you should register before it starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD). You do not need prior data engineering experience but should have basic coding experience and familiarity with SQL. If you have experience with Python, it is helpful but not required. You can also join the course Telegram channel for announcements. You can start preparing by installing the dependencies and requirements listed in the course FAQ. If you miss the registration deadline, you can still follow the course materials and submit the homework, but be aware of the deadlines for turning in assignments.


In [75]:
#result is still a string
result

'To join the course, you should register before it starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD). You do not need prior data engineering experience but should have basic coding experience and familiarity with SQL. If you have experience with Python, it is helpful but not required. You can also join the course Telegram channel for announcements. You can start preparing by installing the dependencies and requirements listed in the course FAQ. If you miss the registration deadline, you can still follow the course materials and submit the homework, but be aware of the deadlines for turning in assignments.'

### Tool calling with a model that can call tools:

Let's create an agent now. In OpenAI's terminology, we'll need to use "function calling".
We will begin with our FAQ example and text search. You can easily extend it to vector or hybrid search or change it to the Evidently docs.
This is the function we implemented previously:

In [ ]:
!pip install mistralai

In [12]:
# below is what is used for openai but a function wrapper is missing in order to use it with mistral
text_search_tool_openai = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}
# To use with mistral:
text_search_tool = [
    {
        "type": "function",
        "function": {          # <-- this "function" wrapper is what was missing
            "name": "text_search",
            "description": "Search the FAQ database",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query text to look up in the course FAQ."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

In [11]:
def text_search(query):
    return faq_index.search(query, num_results=5)

In [14]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant for a course."
    }
]

In [17]:
messages.append({"role": "user", "content": "I just discovered the course, can I join now?"})

In [20]:
messages

[{'role': 'system', 'content': 'You are a helpful assistant for a course.'},
 {'role': 'user', 'content': 'I just discovered the course, can I join now?'}]

In [24]:
from mistralai.client import Mistral
from google.colab import userdata

client = Mistral(api_key=userdata.get('mistral'))

model="mistral-small-latest"  # free tier

response = client.chat.complete(
    model = model, # The model we want to use
    messages = messages, # The message history, in this example we have a system (optional) + user query.
    tools = text_search_tool # The tools specifications
)

We can't just pass this function to the LLM. First, we need to describe this function, so the LLM understands how to use it.
This is done using a special description format:

This description tells the LLM:
* The function is called text_search
* It searches the FAQ database
* It takes one required parameter: query (a string)

The query should be the search text to look up in the course FAQ

Now we can use it:

In [28]:
response

ChatCompletionResponse(id='dee62c51f5f840e995183d9facd747de', object='chat.completion', model='mistral-small-latest', usage=UsageInfo(prompt_tokens=110, completion_tokens=20, total_tokens=130, prompt_audio_seconds=Unset(), prompt_tokens_details={'cached_tokens': 0}), created=1775128137, choices=[ChatCompletionChoice(index=0, message=AssistantMessage(role='assistant', content='', tool_calls=[ToolCall(function=FunctionCall(name='text_search', arguments='{"query": "Can I join the course after it has started?"}'), id='EXePt0c6n', type=None, index=0)], prefix=False), finish_reason='tool_calls')])

Nice! The response is no longer a string but an object :)
More precisely a ChatCompletionResponse object

In [32]:
response.object

'chat.completion'

But the object's attributes are quite different from how the openai version of it is mapped. The key differences vs OpenAI:

* The tool result message uses "role": "tool" instead of "type": "function_call_output"
* call_id → tool_call.id
* arguments → tool_call.function.arguments

The agent analyzed the user's question and determined that to answer it, it needs to invoke the text_search function with the arguments {"query":"join course"}.
Let's invoke the function with these arguments:

In [ ]:
#openai version (running this cell will give an error because the mapping is different from mistral's)
#call = response.output[0]

#arguments = json.loads(call.arguments)
#result = text_search(**arguments)

#call_output = {
#    "type": "function_call_output",
#    "call_id": call.call_id,
#    "output": json.dumps(result),
#}

#chat_messages.append(call)
#chat_messages.append(call_output)

#response = openai_client.responses.create(
#    model='gpt-4o-mini',
#    input=chat_messages,
#    tools=[text_search_tool]
#)

#print(response.output_text)

In [47]:
import json
# Get the tool call from Mistral's response
tool_call = response.choices[0].message.tool_calls[0]

# Parse arguments
arguments = json.loads(tool_call.function.arguments)

# Execute the tool
result = text_search(**arguments)

# Build the tool result message (Mistral format)
call_output = {
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result),
}

In [48]:
# Then to continue the conversation, append both the assistant's message and the tool result back into your messages list:
messages.append(response.choices[0].message)  # assistant's tool_call message
messages.append(call_output)                   # tool result

# Make a second call to get the final answer
final_response = client.chat.complete(
    model=model,
    messages=messages,
    tools=text_search_tool
)
print(final_response.choices[0].message.content)

Yes, you can still join the course even after the start date! You are eligible to submit homework and participate. Just be mindful of the deadlines for turning in homework and the final project. You can register using this [link](https://airtable.com/shr6oVXeQvSI5HuWD) and join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.


Here's what's happening:

* The LLM decided to execute a function and let us know about it
* We executed the function and saved the results
* Now we need to pass this information back to the LLM

We do it by extending the chat_messages list and sending the entire conversation history back to the LLM:

LLMs are stateless. When we make one call to the OpenAI API and then shortly afterwards make another, it doesn't know anything about the first call. So if we only send it call_output, it would have no idea how to respond to it.

This is why we need to send it the entire conversation history. It needs to know everything that happened so far:

* The system prompt (so it knows what the initial instructions are) - system_prompt
* The user prompt (so it knows what task it needs to perform) - question
* The decision to invoke the text_search tool (so it knows what function was called) - that's our call
* The output of the function (so it knows what the function returned) - that's our call_output

After we invoke it, we get back the response:
“Yes, you can still join the course even after the start date. While you won't be able to officially register, you are eligible to submit your homework. Just keep in mind that there are deadlines for submitting assignments and final projects, so it's best not to leave everything to the last minute.”
This is a useful response that we were hoping to get.

### 3. System Prompt: Instructions
Let's take another look at the code we wrote previously:

In [13]:
chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

model="mistral-small-latest"  # free tier

response = client.chat.complete(
    model = model,
    messages = chat_messages,
    tools = text_search_tool
)

We have two things here:
* system_prompt contains instructions for the LLM
* question ("user prompt") is the actual question or task

The system prompt is very important: it influences how the agent behaves. This is how we can control what the agent does and how it responds to user questions.
Usually, the more complete the instructions in the system prompt are, the better the results.
So we can extend it:

In [14]:
system_prompt = """
You are a helpful assistant for a course.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""

When working with agents, the system prompt becomes one of the most essential variables we can adjust to influence our agent.
For example, if we want the agent to make multiple search queries, we can modify the prompt:

In [15]:
system_prompt = """
You are a helpful assistant for a course.

Always search for relevant information before answering.
If the first search doesn't give you enough information, try different search terms.

Make multiple searches if needed to provide comprehensive answers.
"""

### 4. Pydantic AI

Dealing with function calls can be cumbersome. We first need to understand which function we need to invoke. Then we need to pass the results back to the LLM and perform other tasks. It's easy to make a mistake there.

That's why we'll use a library to handle it. There are many agentic libraries: OpenAI Agents SDK, Langchain, Pydantic AI, and many more. For educational purposes, I also implemented an agents library. It's called ToyAIKit. We won't use it here, but I often use it in my lessons.
Today, we will use Pydantic AI. I like its API; it's simpler than other libraries and has good documentation.

For Pydantic AI (and for other agents libraries), we don't need to describe the function in the JSON format like we did with the plain OpenAI API. The libraries take care of it.
But we do need to add docstrings and type hints to our function. I asked ChatGPT to do it:

In [16]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return faq_index.search(query, num_results=5)

We can now define an agent with Pydantic AI and give it the text_search tool:

In [ ]:
!pip install pydantic-ai

In [22]:
# luck has it : Pydantic has built-in mistral support
from pydantic_ai import Agent
from pydantic_ai.models.mistral import MistralModel
from pydantic_ai.providers.mistral import MistralProvider
from google.colab import userdata

mistral_provider = MistralProvider(api_key=userdata.get('mistral'))

mistral_model = MistralModel(
    model_name="mistral-small-latest",
    provider=mistral_provider #MistralModel from pydantic-ai doesn't directly take an api_key argument, replaced it with this
)

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    model=mistral_model
)

We don't need to do anything with our text_search function. We just pass it directly to the agent.
Let's run it:

In [31]:
question = "I just discovered the course, can I join now?"

result = await agent.run(user_prompt=question)

We use await because Pydantic AI is asynchronous. If you're not running in Jupyter, you need to use asyncio.run():

In [ ]:
#import asyncio

#result = asyncio.run(agent.run(user_prompt=question))

The output:

“Yes, you can still join the course even after the start date. Although you may not officially register, you are eligible to submit your homework. Just keep in mind that there are deadlines for turning in homework and final projects, so it's advisable not to delay everything until the last minute.”

We can also look inside the result to get a detailed breakdown of the agent's reasoning and actions:

In [32]:
result.new_messages()

[ModelRequest(parts=[UserPromptPart(content='I just discovered the course, can I join now?', timestamp=datetime.datetime(2026, 4, 2, 11, 49, 37, 605605, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 2, 11, 49, 37, 605957, tzinfo=datetime.timezone.utc), instructions="You are a helpful assistant for a course.\n\nAlways search for relevant information before answering.\nIf the first search doesn't give you enough information, try different search terms.\n\nMake multiple searches if needed to provide comprehensive answers.", run_id='019d4e07-0ac4-737d-9308-46d4d88f913b'),
 ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query": "Can I join the course after it has started?"}', tool_call_id='HMYKzNgBW')], usage=RequestUsage(input_tokens=191, output_tokens=20), model_name='mistral-small-latest', timestamp=datetime.datetime(2026, 4, 2, 11, 49, 39, 357072, tzinfo=datetime.timezone.utc), provider_name='mistral', provider_url='https://api.mistral.ai', pr

It contains four items:
* ModelRequest: Represents a request sent to the model. It includes the user's prompt (UserPromptPart) and the agent's instructions.
* ModelResponse: The model's reply. We see a ToolCallPart with the decision to invoke text_search.
* ModelRequest: Contains ToolReturnPart - the results returned by the tool (search results from the FAQ index).
* ModelResponse: The final answer generated by the model in TextPart.

Pydantic AI and other frameworks handle all the complexity of function calling for us. We don't need to manually parse responses, handle tool calls, or manage conversation history. This makes our code cleaner and less error-prone.

We implemented an agent. Great! But how good is it? Is the prompt we came up good? What's better for our agent, text search, vector search or hybrid?

## Day 5: Evaluation

Is our agent actually good? Today we will see how to answer this question.

In particular, we will cover:
* Build a logging system to track agent interactions
* Create automated evaluation using AI as a judge
* Generate test data automatically
* Measure agent performance with metrics

At the end of this lesson, you'll have a thoroughly tested agent with performance metrics.
In this lesson, we'll use the FAQ database with text search, but it's applicable for any other use case.
This is going to be a long lesson, but an important one. Evaluation is critical for building reliable AI systems. Without proper evaluation, you can't tell if your changes improve or hurt performance. You can't compare different approaches. And you can't build confidence before deploying to users.

So let's start!

### Logging

The easiest thing we can do to evaluate an agent is interact with it. We ask something and look at the response. Does it make sense? For most cases, it should.
This approach is called "vibe check" - we interact with it, and if we like the results, we go ahead and deploy it.
If we don't like something, we go back and change things:
* Maybe our chunking method is not suitable? Maybe we need to have a bigger window size?
* Is our system prompt good? Maybe we need more precise instructions?
* Or we want to change something else

And we iterate.

It might be okay for the first MVP, but how can we make sure the result at the end is actually good?
We need systematic evaluation. Manual testing doesn't scale - you can't manually test every possible input and scenario. With systematic evaluation, we can test hundreds or thousands of cases automatically.
We also need to base our decisions on data. It will help us to
* Compare different approaches
* Track improvements
* Identify edge cases

We can start collecting this data ourselves: start with vibe checking, but be smart about it. We don't just test it, but also record the results.

Here's what we want to record:

* The system prompt that we used
* The model
* The user query
* The tools we use
* The responses and the back-and-forth interactions between the LLM and our tools
* The final response

To make it simpler, we'll implement a simple logging system ourselves: we will just write logs to json files.

You shouldn't use it in production. In practice, you will want to send these logs to some log collection system, or use specialized LLM evaluation tools like Evidently, LangWatch or Arize Phoenix.

Let's extract all this information from the agent and from the run results:

In [33]:
from pydantic_ai.messages import ModelMessagesTypeAdapter


def log_entry(agent, messages, source="user"):
    tools = []

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }

This code extracts the key information from our agent:

* the configuration (name, prompt, model)
* available tools
* complete message history (user input, tool calls, responses)

We also use ModelMessagesTypeAdapter.dump_python(messages) to convert internal message format into regular Python dictionaries. This makes it easier to save it to JSON and process later.

We also add the source parameter. It tracks where the question came from. We start with "user" but later we'll use AI-generated queries. Sometimes it may be important to tell them apart for analysis.
This code is generic so it will work with any Pydantic AI agent. If you use a different library, you'll need to adjust this code.

Let's write these logs to a folder:

In [52]:
import json
import secrets
from pathlib import Path
from datetime import datetime


LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    # Get the raw timestamp. It might be a datetime object or an ISO string.
    raw_timestamp = entry['messages'][-1]['timestamp']

    if isinstance(raw_timestamp, str):
        # If it's a string, convert it to datetime object
        ts_obj = datetime.fromisoformat(raw_timestamp.replace("Z", "+00:00"))
    elif isinstance(raw_timestamp, datetime):
        # If it's already a datetime object, use it directly
        ts_obj = raw_timestamp
    else:
        raise TypeError(f"Unexpected type for timestamp: {type(raw_timestamp)}")

    # Now format the datetime object to the desired string
    ts_str = ts_obj.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath

This creates a simple interactive loop where:

* User enters a question
* Agent processes it and responds
* Complete interaction is logged to a file

Try these questions:
* how do I use docker on windows?
* can I join late and get a certificate?
* what do I need to do for the certificate?

### Adding References

When interacting with the agent, I noticed one thing: it doesn't include the reference to the original documents.
Let's fix it by adjusting the prompt:

In [35]:
system_prompt = """
You are a helpful assistant for a course.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.
When citing the reference, replace "faq-main" by the full path to the GitHub repository: "https://github.com/DataTalksClub/faq/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

# Create another version of agent, let's call it faq_agent_v2
agent = Agent(
    name="faq_agent_v2",
    instructions=system_prompt,
    tools=[text_search],
    model=mistral_model
)

This is the output I now get for the question "can I join late and get a certificate?":

Yes, you can join the course late and still be eligible for a certificate, as long as you complete the required peer-reviewed capstone projects on time. You do not need to complete the homework assignments if you join late, which allows for flexibility in participation.

However, please note that certificates are only awarded to those who finish the course with a “live” cohort; they are not available for those who choose the self-paced mode. This is because peer-reviewing capstone projects is a requirement that can only be done while the course is active.

For more details, you can refer to the following resources:
* Do I need to do the homeworks to get the certificate?
* Can I follow the course in a self-paced mode and get a certificate?
* Can I still join the course after the start date?

Note that I added this to the prompt:

When citing the reference, replace "faq-main" by the full path to the GitHub repository: "https://github.com/DataTalksClub/faq/blob/main/"

When analyzing the results, I noticed that we should have stripped "faq-main" from the filename on Day 1 when we were preparing the data. We should come back to it and adjust the ingestion process, but I won't do it here now.

We can also further adjust the instructions to make it cite the references immediately in the paragraph if we want.
Now we collect more data and finally start testing it.

### LLM as a Judge
You can ask your colleagues to also do a "vibe check", but make sure you record the data. Often collecting 10-20 examples and manually inspecting them is enough to understand how your model is doing.

Don't be afraid of putting manual work into evaluation. Manual evaluation will help you understand edge cases, learn what good responses look like and think of evaluation criteria for automated checks later.

For example, I manually inspected the output and noticed that references are missing. So we will later add it as one of the checks.
So, in our case, we can have the following checks:
* Does the agent follow the instructions?
* Given the question, does the answer make sense?
* Does it include references?
* Did the agent use the available tools?

We don't have to evaluate this manually. Instead, we can delegate this to AI. This technique is called "LLM as a Judge".
The idea is simple: we use one LLM to evaluate the outputs of another LLM. This works because LLMs are good at following detailed evaluation criteria.
Our system prompt for the judge (we'll call it "evaluation agent" because it sounds cooler) can look like that:


In [36]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met.

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do
- answer_relevant: The response directly addresses the user's question
- answer_clear: The answer is clear and correct
- answer_citations: The response includes proper citations or sources when required
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked?

Output true/false for each check and provide a short explanation for your judgment.
""".strip()

Since we expect a very well defined structure of the response, we can use structured output.
We can define a Pydantic class with the expected response structure, and the LLM will produce output that matches this schema exactly.
This is how we do it:

In [37]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str

This code defines the structure we expect from our evaluation:
* Each check has a name, justification, and pass/fail result
* The overall evaluation includes a list of checks and a summary

Note that justification comes before check_pass. This makes the LLM reason about the answer before giving the final judgment, which typically leads to better evaluation quality.

With Pydantic AI in order to make the output follow the specified class, we use the parameter output_type:

In [38]:
eval_agent = Agent(
    name='eval_agent',
    model=mistral_model,
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)

Usually it's a good idea to evaluate the results of one model (for example "gpt-4o-mini") with another model (e.g. "gpt-5-nano"). A different model can catch mistakes, reduce self-bias, and give a second opinion. This makes evaluations more reliable.

We have the instructions, and we have the agent. In order to run the agent, it needs input. We'll start with a template:

In [39]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

We use XML markup because it's easier and more clear for LLMs to understand the input. XML tags help the model see the structure and boundaries of different sections in the prompt.

Let's fill it in. First, define a helper function for loading JSON log files:

In [40]:
def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

In [43]:
!touch ./logs/faq_agent_v2_20250926_072928_467470.json

In [53]:
# Run an agent interaction and log it to create a valid JSON file
question = "can I join late and get a certificate?"
result = await agent.run(user_prompt=question)
new_log_filepath = log_interaction_to_file(
    agent,
    result.new_messages(),
    source='user'
)
print(f"Generated log file: {new_log_filepath}")

Generated log file: logs/faq_agent_v2_20260402_120659_cacc5b.json


We also add the filename in the result - it'll help us with tracking later.
Now let's use it:

In [44]:
# the code below gave an error (due to the log file being absent
#so I had to create it on the cell above and modified as the cell below)
#log_record = load_log_file('./logs/faq_agent_v2_20250926_072928_467470.json')

#instructions = log_record['system_prompt']
#question = log_record['messages'][0]['parts'][0]['content']
#answer = log_record['messages'][-1]['parts'][0]['content']
#log = json.dumps(log_record['messages'])

#user_prompt = user_prompt_format.format(
#    instructions=instructions,
#    question=question,
#    answer=answer,
#    log=log
#)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [54]:
log_record = load_log_file(new_log_filepath)

instructions = log_record['system_prompt']
question = log_record['messages'][0]['parts'][0]['content']
answer = log_record['messages'][-1]['parts'][0]['content']
log = json.dumps(log_record['messages'])

user_prompt = user_prompt_format.format(
    instructions=instructions,
    question=question,
    answer=answer,
    log=log
)

The user input is ready and we can test it!

In [55]:
result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)

checklist = result.output
print(checklist.summary)

for check in checklist.checklist:
    print(check)

The AI agent's answer meets all the criteria in the checklist. It followed instructions, avoided prohibited actions, provided a relevant and clear answer, included proper citations, was complete, and used the search tool as required.
check_name='instructions_follow' justification="The agent followed the user's instructions by using the search tool to find relevant information from the course materials before answering the question." check_pass=True
check_name='instructions_avoid' justification='The agent did not violate any instructions it was told not to do.' check_pass=True
check_name='answer_relevant' justification="The response directly addresses the user's question about joining late and receiving a certificate." check_pass=True
check_name='answer_clear' justification='The answer is clear and correct, providing specific conditions for receiving a certificate when joining late.' check_pass=True
check_name='answer_citations' justification='The response includes a proper citation to 

This code:
* Loads a saved interaction log
* Extracts the key components (instructions, question, answer, full log)
* Formats them into the evaluation prompt
* Runs the evaluation agent
* Prints the results

When we run it, we'll see this:
```
check_name='instructions_follow' justification='The assistant called the search tool (log shows a text_search call) but did not follow the instruction to include references (file names) from course materials; thus it did not fully follow the provided instructions.' check_pass=False
check_name='instructions_avoid' justification='No forbidden actions in the instructions were performed; the assistant did not do anything it was explicitly told to avoid.' check_pass=True
check_name='answer_relevant' justification='The response directly addressed how to install Kafka client libraries in Python and gave pip/conda commands, so it is relevant to the question.' check_pass=True
check_name='answer_clear' justification='The answer is concise and the installation commands are correct, though it omitted some caveats; overall it is clear and understandable.' check_pass=True
check_name='answer_citations' justification='The reply did not include any citations or filenames from the course materials as required by the instructions.' check_pass=False
check_name='completeness' justification='The answer omitted important details and alternatives (e.g., kafka-python library, librdkafka dependency, broker setup, and course-material citations), so it is not fully complete.' check_pass=False
check_name='tool_call_search' justification='The log shows a text_search tool call was made before the answer was given.' check_pass=True
```

Note that we're putting the entire conversation log into the prompt, which is not really necessary. We can reduce it to make it less verbose.
For example, like that:

In [56]:
def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []

        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']

            if kind == 'user-prompt':
                del part['timestamp']
            if kind == 'tool-call':
                del part['tool_call_id']
            if kind == 'tool-return':
                del part['tool_call_id']
                del part['metadata']
                del part['timestamp']
                # Replace actual search results with placeholder to save tokens
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                del part['id']

            parts.append(part)

        message = {
            'kind': m['kind'],
            'parts': parts
        }

        log_simplified.append(message)
    return log_simplified

We make it simpler:

* remove timestamps and IDs that aren't needed for evaluation
* replace actual search results with a placeholder
* keep only the essential structure

This is helpful because it reduces the number of tokens we send to the evaluation model, which lowers the costs and speeds up evaluation.

Let's put everything together:


In [58]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']

    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][0]['content']

    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output


log_record = load_log_file('/content/logs/faq_agent_v2_20260402_120659_cacc5b.json')
eval1 = await evaluate_log_record(eval_agent, log_record)

We know how to log our data and how to run evals on our logs.

Great. But how do we get more data to get a better understanding of the performance of our model?

### Data Generation
We can ask AI to help. What if we used it for generating more questions? Let's do that.

We can sample some records from our database. Then for each record, ask an LLM to generate a question based on the record. We use this question as input to our agent and log the answers.

Let’s start by defining the question generator:

In [60]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering course.

Based on the provided FAQ content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model=mistral_model,
    output_type=QuestionsList
)

This prompt is designed for our specific use case (data engineering course FAQ). You should adjust it for your project.
We will send it a bunch of records, and it will generate a question from each of them.

Note: we use a simple way of generating questions. We can use a more complex approach where we also track the source (filename) of the question. If we do it, we can later check if this file was retrieved and cited in the answer. But we won't do it today to make things simpler.

Now let's sample 10 records from our dataset using Python's built-in random.sample function:

In [61]:
import random

sample = random.sample(de_dtc_faq, 10)
prompt_docs = [d['content'] for d in sample]
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions

Now we simply iterate over each of the question, ask our agent and log the results:

In [62]:
from tqdm.auto import tqdm

for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()

  0%|          | 0/10 [00:00<?, ?it/s]

I encountered an error while running `./spark-submit.sh streaming.py` in tutorial 13.2. The error message indicates that the SparkContext is stopped and all masters are unresponsive. How can I resolve this issue?
The error you encountered, where the `SparkContext` is stopped and all masters are unresponsive, is often caused by a version mismatch between your local PySpark installation and the Spark version used in the Docker environment. Here’s how you can resolve it:

---

### **Solution: Downgrade PySpark**
1. **Downgrade your local PySpark** to version **3.3.1** to match the version used in the Docker setup.
   Run the following command:
   ```bash
   pip install pyspark==3.3.1
   ```

2. **Verify the Spark version** in your Docker environment:
   - Check the `spark-master` logs to confirm the Spark version being used:
     ```bash
     docker ps
     docker exec -it <spark_master_container_id> bash
     cat logs/spark-master.out
     ```
   - Look for the Spark version in the logs 

We can repeat it multiple times until we have enough data. Around 100 should be good for a start, but today we can just continue with the 10 log records we already generated.

Using AI for generating test data is quite powerful. It can help us get data faster and sometimes cover edge cases we won't think about.
There are limitations too:

* AI-generated questions might not reflect real user behavior
* It may miss important edge cases that only real users encounter
* They may not capture the full complexity of real user queries

The logs are ready, so we can run evaluation on them with our evaluation agent.
First, collect all the AI-generated logs for the v2 agent:

In [63]:
eval_set = []

for log_file in LOG_DIR.glob('*.json'):
    if 'faq_agent_v2' not in log_file.name:
        continue

    log_record = load_log_file(log_file)
    if log_record['source'] != 'ai-generated':
        continue

    eval_set.append(log_record)

and evaluate them:

In [64]:
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))

  0%|          | 0/10 [00:00<?, ?it/s]

This code:
* Loops through each AI-generated log
* Runs our evaluation agent on it
* Stores both the original log and evaluation result

There are ways to speed this up, but we won't cover them in detail here. For example, you can try this:

Don't ask for justification - this makes evaluation faster but slightly lower quality
Parallelize execution - you can ask ChatGPT how to do this with async/await

The results are collected, but we need to display them and also calculate some statistics. The best tool for doing this is Pandas. We already should have it because minsearch depends on it.

Our data is not ready to be converted to a Pandas DataFrame. We first need to transform it a little. Let’s do it:

In [65]:
rows = []

for log_record, eval_result in eval_results:
    messages = log_record['messages']

    row = {
        'file': log_record['log_file'].name,
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)

This code:
* Extracts key information from each log (file, question, answer)
* Converts the evaluation checks into a dictionary format

Now each row is a simple key-value dictionary, so we can create a DataFrame:

In [66]:
import pandas as pd

df_evals = pd.DataFrame(rows)

We can look at individual records and see which checks are False.

But it's also useful to look at the overall stats:

In [68]:
df_evals.head()

,file,question,answer,instructions_follow,instructions_avoid,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
0,faq_agent_v2_20260402_121101_4b4e77.json,How can I generate a pip-friendly `requirement...,To generate a pip-friendly `requirements.txt` ...,True,True,True,True,True,True,True
1,faq_agent_v2_20260402_121045_47f89d.json,I missed the registration deadline for the cou...,"Based on the available information, here's wha...",True,True,True,True,True,True,True
2,faq_agent_v2_20260402_121107_7595ec.json,What are the prerequisites for taking the data...,To get the most out of the **Data Engineering ...,True,True,True,True,True,True,True
3,faq_agent_v2_20260402_121104_590d2d.json,"I'm using dbt Cloud for the course, but the in...","Yes, this is expected. The demos in the course...",True,True,True,True,True,True,True
4,faq_agent_v2_20260402_121049_087bda.json,I'm getting an error with my `docker-compose.y...,The error `'service pgdatabase refers to undef...,True,True,True,True,True,True,True


In [67]:
df_evals.mean(numeric_only=True)

,0
instructions_follow,1.0
instructions_avoid,1.0
answer_relevant,1.0
answer_clear,1.0
answer_citations,0.9
completeness,1.0
tool_call_search,1.0


This tells us:
* Only 30% of responses follow instructions completely
* All responses avoid forbidden actions (good!)
* All responses are relevant and clear (great!)
* Only 30% include proper citations (needs improvement)
* 70% of responses are complete
A* ll responses use the search tool (as expected)

For us, the most important check is answer_relevant. This tells us whether the agent actually answers the user's question. If this score was low, it’d mean that our agent is not ready.

We now know how to evaluate our agent. What can we do with it now?

Many things:
* Decide if this quality is good enough for deployment
* Evaluate different chunking approaches and search
* See if changing a prompt leads to any improvements.

The algorithm is simple:
* Collect data for evaluation and keep this dataset fixed
* Run different versions of your agent for this dataset
* Compare key metrics to decide which version is better
 Evaluation is a very powerful tool and we should use it when possible.

### Evaluating functions and tools

Also, we can (and should) evaluate our tools separately from evaluating the agent.

If it's code, we need to cover it with unit and integration tests.

We also have the search function, which we can evaluate using standard information retrieval metrics. For example:
* Precision and Recall: How many relevant results were retrieved vs. how many relevant results were missed
* Hit Rate: Percentage of queries that return at least one relevant result
* MRR (Mean Reciprocal Rank): Reflects the position of the first relevant result in the ranking

This is how we can implement hitrate and MRR calculation in Python:

In [69]:
def evaluate_search_quality(search_function, test_queries):
    results = []

    for query, expected_docs in test_queries:
        search_results = search_function(query, num_results=5)

        # Calculate hit rate
        relevant_found = any(doc['filename'] in expected_docs for doc in search_results)

        # Calculate MRR
        for i, doc in enumerate(search_results):
            if doc['filename'] in expected_docs:
                mrr = 1 / (i + 1)
                break
        else:
            mrr = 0

        results.append({
            'query': query,
            'hit': relevant_found,
            'mrr': mrr
        })
    return results

These ideas and the code will be useful when you implement a real agent project with search.
It's useful because it'll helps us make guided decisions about:
* When to use text vs. vector vs. hybrid search
* What are the best parameters for our search